In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/processed/featured_data.csv')
print(f"Loaded: {df.shape[0]:,} rows | {df.shape[1]:,} columns")

target = 'isFraud'

# ============================================================
# STEP 1: FIX TARGET LEAKAGE IN RISK SCORE FEATURES
# Problem: product_risk_score, card4_risk_score, card6_risk_score were computed using the FULL dataset including
# what will become the test set — this leaks the target into the features
# Fix: Drop the leaked versions, recompute using ONLY training data after the split
# ============================================================

leaked_features = ['product_risk_score', 'card4_risk_score', 'card6_risk_score']
df.drop(columns=leaked_features, inplace=True)
print(f"\n[STEP 1] Dropped leaked features: {leaked_features}")
print("Will recompute these using train-only data after split.")

Loaded: 590,540 rows | 224 columns

[STEP 1] Dropped leaked features: ['product_risk_score', 'card4_risk_score', 'card6_risk_score']
Will recompute these using train-only data after split.


In [3]:
# ============================================================
# STEP 2: TRAIN-TEST SPLIT (STRATIFIED)
# Reason: Stratify on target to preserve 3.5% fraud ratio
#         in both train and test sets
# ============================================================

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

print(f"\n[STEP 2] Train-Test Split")
print(f"  Train shape: {X_train.shape}")
print(f"  Test shape : {X_test.shape}")
print(f"  Train fraud rate: {y_train.mean()*100:.2f}%")
print(f"  Test fraud rate : {y_test.mean()*100:.2f}%")


[STEP 2] Train-Test Split
  Train shape: (472432, 220)
  Test shape : (118108, 220)
  Train fraud rate: 3.50%
  Test fraud rate : 3.50%


In [4]:
# ============================================================
# STEP 3: RECOMPUTE RISK SCORES — TRAIN-ONLY (LEAK-FREE)
# Reason: Compute fraud rate per category using ONLY training data, then map to both train and test.
# Test set categories unseen in train get the global training fraud rate (safe fallback)
# ============================================================

global_fraud_rate = y_train.mean()

def safe_target_encode(train_col, train_target, test_col, smoothing=10):
    """
    Target encoding with smoothing to avoid overfitting
    on categories with very few samples.
    """
    temp = pd.DataFrame({'col': train_col, 'target': train_target})
    agg = temp.groupby('col')['target'].agg(['mean', 'count'])

    # Smoothing: blend category mean with global mean
    # based on how many samples support that category
    smoothed_mean = ((agg['mean'] * agg['count']) + (global_fraud_rate * smoothing)) / (agg['count'] + smoothing)

    mapping = smoothed_mean.to_dict()

    train_encoded = train_col.map(mapping).fillna(global_fraud_rate)
    test_encoded  = test_col.map(mapping).fillna(global_fraud_rate)

    return train_encoded, test_encoded

# Recompute for ProductCD, card4, card6
for col in ['ProductCD', 'card4', 'card6']:
    train_enc, test_enc = safe_target_encode(X_train[col], y_train, X_test[col])
    X_train[f'{col}_risk_score'] = train_enc
    X_test[f'{col}_risk_score']  = test_enc

print(f"\n[STEP 3] Leak-free risk scores recomputed:")
print(f"  ProductCD_risk_score | card4_risk_score | card6_risk_score")
print(f"  Train mean: {X_train['ProductCD_risk_score'].mean():.4f}")
print(f"  Test mean : {X_test['ProductCD_risk_score'].mean():.4f}")

# Recompute interaction feature that depended on leaked scores
X_train['product_card_risk'] = X_train['ProductCD_risk_score'] * X_train['card6_risk_score']
X_test['product_card_risk']  = X_test['ProductCD_risk_score'] * X_test['card6_risk_score']


[STEP 3] Leak-free risk scores recomputed:
  ProductCD_risk_score | card4_risk_score | card6_risk_score
  Train mean: 0.0350
  Test mean : 0.0348


In [5]:

# ============================================================
# STEP 4: BASELINE MODEL — LOGISTIC REGRESSION
# Reason: Always establish a baseline before complex models If XGBoost doesn't beat this by a wide margin,
# something is wrong with feature engineering
# ============================================================

# Scale features for logistic regression (tree models don't need this)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

baseline = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Handles 3.5% imbalance
    random_state=42
)
baseline.fit(X_train_scaled, y_train)

baseline_train_pred = baseline.predict_proba(X_train_scaled)[:, 1]
baseline_test_pred  = baseline.predict_proba(X_test_scaled)[:, 1]

from sklearn.metrics import roc_auc_score
baseline_train_auc = roc_auc_score(y_train, baseline_train_pred)
baseline_test_auc  = roc_auc_score(y_test, baseline_test_pred)

print(f"\n[STEP 4] Baseline Logistic Regression")
print(f"  Train AUC: {baseline_train_auc:.4f}")
print(f"  Test AUC : {baseline_test_auc:.4f}")


[STEP 4] Baseline Logistic Regression
  Train AUC: 0.8524
  Test AUC : 0.8477


In [6]:

# ============================================================
# STEP 5: MAIN MODEL — XGBOOST
# Reason: XGBoost handles:
#   - Mixed feature types without scaling
#   - Class imbalance via scale_pos_weight
#   - Non-linear interactions automatically
#   - Missing values natively (though we have none)
# ============================================================

# Calculate scale_pos_weight for imbalance
# Formula: count(negative) / count(positive)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"\n[STEP 5] XGBoost Configuration")
print(f"  scale_pos_weight: {scale_pos_weight:.2f}")
print(f"  (This tells XGBoost fraud cases are {scale_pos_weight:.0f}x rarer)")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',          # AUC-PR better than AUC-ROC for imbalance
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=20
)

xgb_model.fit(X_train, y_train,eval_set=[(X_test, y_test)],verbose=50)

xgb_train_pred = xgb_model.predict_proba(X_train)[:, 1]
xgb_test_pred  = xgb_model.predict_proba(X_test)[:, 1]

xgb_train_auc = roc_auc_score(y_train, xgb_train_pred)
xgb_test_auc  = roc_auc_score(y_test, xgb_test_pred)

print(f"\n[STEP 5] XGBoost Results")
print(f"  Train AUC: {xgb_train_auc:.4f}")
print(f"  Test AUC : {xgb_test_auc:.4f}")


[STEP 5] XGBoost Configuration
  scale_pos_weight: 27.58
  (This tells XGBoost fraud cases are 28x rarer)
[0]	validation_0-aucpr:0.28210
[50]	validation_0-aucpr:0.51852
[100]	validation_0-aucpr:0.55937
[150]	validation_0-aucpr:0.58645
[200]	validation_0-aucpr:0.60464
[250]	validation_0-aucpr:0.62183
[299]	validation_0-aucpr:0.63491

[STEP 5] XGBoost Results
  Train AUC: 0.9580
  Test AUC : 0.9399


In [9]:
# ============================================================
# STEP 6: MODEL COMPARISON
# ============================================================

print("""========== MODEL COMPARISON ==========""")
print("Model                   | Train AUC              | Test AUC")
print("------------------------------------------------")
print(f"Logistic Regression    | {baseline_train_auc:.4f}    | {baseline_test_auc:.4f}")
print(f"XGBoost                | {xgb_train_auc:.4f}    | {xgb_test_auc:.4f}")
print("========================================")

print(f"Improvement: {(xgb_test_auc - baseline_test_auc)*100:.2f} percentage points")

# Check for overfitting
train_test_gap = xgb_train_auc - xgb_test_auc
if train_test_gap > 0.05:
    print(f"WARNING: Train-Test AUC gap is {train_test_gap:.4f} — possible overfitting")
else:
    print(f"Train-Test AUC gap: {train_test_gap:.4f} — acceptable, no major overfitting")

========== MODEL COMPARISON ==========
Model                   | Train AUC              | Test AUC
------------------------------------------------
Logistic Regression    | 0.8524    | 0.8477
XGBoost                | 0.9580    | 0.9399
Improvement: 9.22 percentage points
Train-Test AUC gap: 0.0181 — acceptable, no major overfitting


In [10]:
# ============================================================
# STEP 7: SAVE MODELS & DATA SPLITS
# ============================================================

import os
os.makedirs('models', exist_ok=True)

joblib.dump(xgb_model, 'models/xgboost_fraud_model.pkl')
joblib.dump(baseline, 'models/baseline_logistic_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print(f"\nModels saved to models/ directory")
print(f"Train/test splits saved to data/processed/")


Models saved to models/ directory
Train/test splits saved to data/processed/
